# Transfer Learning (VGG-16)

In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [2]:
torch.manual_seed(42)

In [3]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: mps


In [4]:
df = pd.read_csv('data/fashion-mnist_train.csv')
df.head(2)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [7]:
#Transformations
from torchvision import transforms
custom_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [23]:
from PIL import Image
import numpy as np

class CustomDataset(Dataset):
    def __init__(self, features, label, transforms):
        self.features = features
        self.label = label
        self.transform = transforms
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        image = self.features[index].reshape(28,28)
        image = image.astype(np.uint8)
        image = np.stack([image]*3, axis=-1)

        image = Image.fromarray(image)

        image = self.transform(image)

        return image, torch.tensor(self.label[index], dtype=torch.long)

In [24]:
train_dataset = CustomDataset(X_train, y_train, transforms=custom_transform)
#create test dataset
test_dataset = CustomDataset(X_test,y_test, transforms=custom_transform)

In [25]:
#create train and test loader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [12]:
import torchvision.models as models
vgg16 = models.vgg16(pretrained=True)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /Users/atharvakhandelwal/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [02:44<00:00, 3.37MB/s] 


In [13]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [26]:
for param in vgg16.features.parameters():
    param.requires_grad = False


In [27]:
vgg16.classifier = nn.Sequential(
    nn.Linear(25088, 1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024,512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512,10)
)

In [28]:
vgg16 = vgg16.to(device)

In [29]:
#set learning rate and epochs
epochs = 4
learning_rate = 0.0001

In [30]:
criterion = nn.CrossEntropyLoss()

#optimizer
optimizer = optim.Adam(vgg16.classifier.parameters(), lr=learning_rate)

In [32]:
# #training loop

# for epoch in range(epochs):
#     total_epoch_loss = 0
#     for batch_features, batch_label in train_loader:
#         batch_features, batch_label = batch_features.to(device), batch_label.to(device)
#         #forwardpass
#         outputs = vgg16(batch_features)
#         #calculate loss
#         loss = criterion(outputs, batch_label)
#         #backpass
#         optimizer.zero_grad()
#         loss.backward()
#         #updateparams
#         optimizer.step()

#         total_epoch_loss = total_epoch_loss + loss.item()

#     avg_loss = total_epoch_loss/len(train_loader)
#     print(f'Epoch: {epoch+1}, Loss:{avg_loss:.4f}')
    